In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import math
import operator
import csv

In [2]:
pd.options.display.max_colwidth = 250

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('LazarusNLP/all-indo-e5-small-v4')

In [4]:
# Membaca file madaniyah.csv untuk mendapatkan nomor surah
madaniyah_chapters = []
with open('../madaniyah.csv', 'r') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)  # Melewati header
    for row in csv_reader:
        madaniyah_chapters.append(int(row[1]))
        

sentences = []

for chapter in madaniyah_chapters:

  quran_chapter= pd.read_csv('../Dataset/chapter_' + str(chapter) + '.csv')
  sentence1 = quran_chapter['king_fahd'].tolist()
  sentence2 = quran_chapter['kemenag'].tolist()
  sentence3 = quran_chapter['google_translate'].tolist()

  sentences.append(sentence1)
  sentences.append(sentence2)
  sentences.append(sentence3)

  # print('Chapter - ', str(chapter), ' ', len(eknath_easwaran_chapter))

sentence_new = []
for i in range(0, len(sentences)):
  for j in range(0, len(sentences[i])):
    sentence_new.append(sentences[i][j])

sentence_embeddings = model.encode(sentence_new)

In [5]:
sentence_embeddings.shape

(3897, 384)

In [7]:
import pandas as pd
import csv
from sklearn.metrics.pairwise import cosine_similarity

# Inisialisasi
index_map = []
current_index = 0
df = pd.DataFrame(columns=[
    'Chapter', 'Verse', 'King Fahd', 'Kemenag', 'Google Translator',
    'Fahd - Google', 'Kemenag - Google', 'Fahd - Kemenag'
])

# Bangun peta indeks untuk masing-masing surah
for chapter in madaniyah_chapters:
    df_chapter = pd.read_csv(f'../Dataset/chapter_{chapter}.csv')
    n_verses = len(df_chapter)

    index_map.append({
        'chapter': chapter,
        'start_index': current_index,
        'num_verses': n_verses
    })

    current_index += 3 * n_verses  # 3 versi: king_fahd, kemenag, google

# Loop seluruh surah dan ayat untuk hitung similarity
for mapping in index_map:
    chapter = mapping['chapter']
    start = mapping['start_index']
    n_verses = mapping['num_verses']

    for verse in range(n_verses):
        i_fahd = start + verse
        i_kemenag = start + n_verses + verse
        i_google = start + 2 * n_verses + verse

        # Hitung cosine similarity
        sim_fahd = cosine_similarity([sentence_embeddings[i_fahd]], [sentence_embeddings[i_google]])[0][0]
        sim_kemenag = cosine_similarity([sentence_embeddings[i_kemenag]], [sentence_embeddings[i_google]])[0][0]
        sim_fahd_kemenag = cosine_similarity([sentence_embeddings[i_fahd]], [sentence_embeddings[i_kemenag]])[0][0]

        dict_row = {
            'Chapter': chapter,
            'Verse': verse + 1,
            'King Fahd': sentence_new[i_fahd],
            'Kemenag': sentence_new[i_kemenag],
            'Google Translator': sentence_new[i_google],
            'Fahd - Google': sim_fahd,
            'Kemenag - Google': sim_kemenag,
            'Fahd - Kemenag': sim_fahd_kemenag
        }

        df = pd.concat([df, pd.DataFrame([dict_row])], ignore_index=True)

# Tampilkan hasil akhir
df


/var/folders/p8/hpcxrgfd4wv_j3fp535z1ytc0000gn/T/ipykernel_62495/1074278710.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([dict_row])], ignore_index=True)


,Chapter,Verse,King Fahd,Kemenag,Google Translator,Fahd - Google,Kemenag - Google,Fahd - Kemenag
0,2,1,Alif Lām Mīm.,Alif Lām Mīm.,nyeri,0.010567,0.010567,1.000000
1,2,2,"Kitab (Al-Qur`ān) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Kitab (Alquran) ini tidak ada keraguan padanya; petunjuk bagi mereka yang bertakwa,","Itu adalah Tuhan, tidak ada keraguan ۛ di dalamnya ۛ dipandu kepada orang benar.",0.338157,0.338785,0.962244
2,2,3,"(yaitu) mereka yang beriman kepada yang gaib, yang mendirikan salat, dan menafkahkan sebagian rezeki yang Kami anugerahkan kepada mereka,","(Yaitu) mereka yang beriman kepada yang gaib, melaksanakan salat, dan menginfakkan sebagian rezeki yang Kami berikan kepada mereka,",Mereka yang percaya pada yang tidak terlihat dan mengevaluasi kenaikan,0.354731,0.380840,0.975567
3,2,4,"dan mereka yang beriman kepada Kitab (Al-Qur`ān) yang telah diturunkan kepadamu dan kitab-kitab yang telah diturunkan sebelummu, serta mereka yakin akan adanya (kehidupan) akhirat.","dan mereka beriman kepada (Alquran) yang diturunkan kepadamu (Muhammad) dan (kitab-kitab) yang telah diturunkan sebelum engkau, dan mereka yakin akan adanya akhirat.",Dan mereka yang percaya pada apa yang saya datangi kepada Anda dan apa yang telah saya ungkapkan di hadapan Anda dan dengan kemewahan mereka pasti,0.600687,0.614015,0.920318
4,2,5,"Mereka itulah yang tetap mendapat petunjuk dari Tuhan mereka, dan merekalah orang-orang yang beruntung.","Merekalah yang mendapat petunjuk dari Tuhannya, dan mereka itulah orang-orang yang beruntung.",Itulah bimbingan dari Tuhan mereka 😂 dan mereka yang adalah orang -orang,0.559419,0.552851,0.932781
...,...,...,...,...,...,...,...,...
1294,66,8,"Hai orang-orang yang beriman, bertobatlah kepada Allah dengan tobat yang semurni-murninya, mudah-mudahan Tuhan kamu akan menghapus kesalahan-kesalahanmu dan memasukkanmu ke dalam surga yang mengalir di bawahnya sungai-sungai, pada hari ketika All...","Wahai orang-orang yang beriman! Bertobatlah kepada Allah dengan tobat yang semurni-murninya, mudah-mudahan Tuhan kamu akan menghapus kesalahan-kesalahanmu dan memasukkan kamu ke dalam surga-surga yang mengalir di bawahnya sungai-sungai, pada hari...","O, mereka yang percaya kepada Tuhan, kepada Tuhan, seorang yang bertobat, dan Tuhanmu, semoga Tuhanmu menebusmu hari itu adalah hari ketika Allah tidak mempermalukan Nabi dan mereka yang percaya dengan Dia ۖ Lampu mereka mencari di antara tangan ...",0.688071,0.652938,0.928247
1295,66,9,"Hai Nabi, perangilah orang-orang kafir dan orang-orang munafik dan bersikap keraslah terhadap mereka. Tempat mereka adalah jahanam dan itu adalah seburuk-buruknya tempat kembali.",Wahai Nabi! Perangilah orang-orang kafir dan orang-orang munafik dan bersikap keraslah terhadap mereka. Tempat mereka adalah neraka Jahanam dan itulah seburuk-buruk tempat kembali.,"Oh, nabi, ketidakpercayaan, mereka berdua, dan orang -orang dari mereka.",0.431174,0.383519,0.928341
1296,66,10,"Allah membuat istri Nūḥ dan istri Lūṭ sebagai perumpamaan bagi orang-orang kafir. Keduanya berada di bawah pengawasan dua orang hamba yang saleh di antara hamba-hamba Kami; lalu kedua istri itu berkhianat kepada kedua suaminya, maka kedua suamin...","Allah membuat perumpamaan bagi orang-orang kafir, istri Nuh dan istri Luṭ. Keduanya berada di bawah pengawasan dua orang hamba yang saleh di antara hamba-hamba Kami; lalu kedua istri itu berkhianat kepada kedua suaminya, tetapi kedua suaminya itu...","Tuhan memberi contoh bagi mereka yang tidak percaya, Nuh, dan seorang wanita yang putus asa. Tentang mereka dari Tuhan, sesuatu, dan dikatakan bahwa mereka memasuki api dengan dua tahun.",0.568998,0.620032,0.930484
1297,66,11,"Dan Allah membuat istri Firʻawn perumpamaan bagi orang-orang yang beriman, ketika ia berkata, ""Ya Tuhan-ku, bangunlah untukku sebuah rumah di sisi-Mu dalam surga dan selamatkanlah aku dari Firʻawn dan perbuatannya, dan selamatkanlah aku dari ka...","Dan Allah membuat perumpamaan bag

In [ ]:
# df.to_csv('../Semantic Analysis Results/semantic_analysis.csv', index=False)

In [10]:
import nltk
from nltk.translate.meteor_score import single_meteor_score
import tqdm

nltk.download('punkt')

# DataFrame hasil
df_meteor = pd.DataFrame(columns=['Chapter', 'Verse', 'Fahd - Google', 'Kemenag - Google', 'Fahd - Kemenag'])

# Proses METEOR
for i in tqdm(range(len(df))):
    chapter = df.loc[i, 'Chapter']
    verse = df.loc[i, 'Verse']

    fahd = str(df.loc[i, 'King Fahd'])
    kemenag = str(df.loc[i, 'Kemenag'])
    google = str(df.loc[i, 'Google Translator'])

    try:
        meteor_fahd_google = single_meteor_score(fahd, google)
        meteor_kemenag_google = single_meteor_score(kemenag, google)
        meteor_fahd_kemenag = single_meteor_score(fahd, kemenag)
    except:
        meteor_fahd_google = np.nan
        meteor_kemenag_google = np.nan
        meteor_fahd_kemenag = np.nan

    df_meteor.loc[i] = [chapter, verse, meteor_fahd_google, meteor_kemenag_google, meteor_fahd_kemenag]

# Simpan
df_meteor.to_csv("meteor_similarity.csv", index=False)

[nltk_data] Downloading package punkt to /Users/reginald/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


TypeError: 'module' object is not callable